# Phase 1: Data Ingestion & Clinical Sanitation

**Goal:** Clean the raw diabetic dataset so it can be used in the next phases.

**What I did**
- Fixed the missing values (`?`)
- Dropped the `weight` column
- Removed patients who died
- Removed duplicate rows

**Why:** The raw data is messy. Wrong or missing values would give wrong results later in scraping, EDA and the risk score.

## Setup

**What I did**
- Imported pandas
- Put the file paths and the weight threshold (90%) at the top

**Why:** All settings are in one place, so they are easy to find and change.

In [44]:
import pandas as pd

# File paths
RAW_DATA = '../data/raw/diabetic_data.csv'
ID_MAPPING = '../data/raw/IDs_mapping.csv'
OUT_CLEANED = '../data/processed/01_cleaned.csv'

# Drop a column if more than this % of values are missing
WEIGHT_MISSING_THRESHOLD = 90

## 1. Loading the Data and Fixing Nulls

**What I did**
- Loaded `diabetic_data.csv`
- Read `?` as NaN while loading
- Used `keep_default_na=False` and checked the first rows with `.head()`

**Why**
- This dataset uses `?` for missing values. Pandas would treat it as normal text, so I changed it to NaN.
- `keep_default_na=False` is important. Values like `'None'` (a lab test that was not done) are real answers, not missing data.

In [45]:
# Load data, treat '?' as NaN and keep valid 'None' labels
df = pd.read_csv(RAW_DATA, na_values=['?'], keep_default_na=False, low_memory=False)

rows_raw = len(df)
print('Raw shape:', df.shape)
df.head(10)

Raw shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
5,35754,82637451,Caucasian,Male,[50-60),NaN,2,1,2,3,...,No,Steady,No,No,No,No,No,No,Yes,>30
6,55842,84259809,Caucasian,Male,[60-70),NaN,3,1,2,4,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
7,63768,114882984,Caucasian,Male,[70-80),NaN,1,1,7,5,...,No,No,No,No,No,No,No,No,Yes,>30
8,12522,48330783,Caucasian,Female,[80-90),NaN,2,1,4,13,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
9,15738,63555939,Caucasian,Female,[90-100),NaN,3,3,4,12,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 2. Checking the Data

**What I did**
- Ran `.info()`, `.describe()` and `.columns`
- Looked for columns with the wrong data type

**What I found:** `encounter_id`, `patient_nbr`, `admission_type_id`, `discharge_disposition_id` and `admission_source_id` are stored as numbers (int64).

**Why:** These columns are only labels (IDs), not real numbers. An average of an ID means nothing, so I fix them in step 7.

In [46]:
# Data types and non-null counts
df.info()

# Summary statistics for numeric columns
display(df.describe())

# All column names
print(df.columns.tolist())

# ID columns that are wrongly stored as integers
df[['encounter_id', 'patient_nbr', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id']].dtypes

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      99493 non-null   str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    3197 non-null    str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                61510 non-null   str  
 11  medical_specialty         51817 non-null   str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

,encounter_id,patient_nbr,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,1.017660e+05,1.017660e+05,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,5.433040e+07,2.024006,3.715642,5.754437,4.395987,43.095641,1.339730,16.021844,0.369357,0.197836,0.635566,7.422607
std,1.026403e+08,3.869636e+07,1.445403,5.280166,4.064081,2.985108,19.674362,1.705807,8.127566,1.267265,0.930472,1.262863,1.933600
min,1.252200e+04,1.350000e+02,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,8.496119e+07,2.341322e+07,1.000000,1.000000,1.000000,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,1.523890e+08,4.550514e+07,1.000000,1.000000,7.000000,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,2.302709e+08,8.754595e+07,3.000000,4.000000,7.000000,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,4.438672e+08,1.895026e+08,8.000000,28.000000,25.000000,14.000000,132.000000,6.000000,81.000000,42.000000,76.000000,21.000000,16.000000


['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


encounter_id                int64
patient_nbr                 int64
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
dtype: object

## 3. Finding the 22 Oral Medication Columns

**What I did**
- Made a list of the 22 medication columns
- Used `assert` to check that every column exists

**Why**
- The data dictionary only says "[22 Medications]" and does not name them.
- These columns form the "Oral Medication" group that I use in later phases.
- If a name is wrong, `assert` stops the code straight away.

In [47]:
# The 22 oral medication columns
ORAL_MEDICATIONS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone',
]

# Make sure every column exists
missing_cols = [c for c in ORAL_MEDICATIONS if c not in df.columns]
assert not missing_cols, f'Columns not found: {missing_cols}'
print(len(ORAL_MEDICATIONS), 'oral medication columns found')

22 oral medication columns found


## 4. Missing Values and the Weight Column

**What I did**
- Calculated the missing % of every column
- Dropped `weight` because it is missing in 96.9% of rows (more than the 90% limit)
- Kept `payer_code` (39.6% missing) and `medical_specialty` (49.1% missing)

**Why**
- Filling 97% of a column would mean making up data, so dropping `weight` is safer.
- `payer_code` and `medical_specialty` are below 90% and still useful, so I kept them. I did not fill them. The missing values stay as NaN and can be treated as "Unknown".

**Data Quality Limitation (for the report):** Weight was recorded for only about 3% of encounters, so it cannot be used in the analysis.

In [48]:
# Missing % for every column, highest first
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(10).round(2))

# Drop weight if it is mostly missing
weight_missing = missing_pct['weight']
if weight_missing > WEIGHT_MISSING_THRESHOLD:
    df = df.drop(columns=['weight'])
    print(f'weight dropped ({weight_missing:.1f}% missing)')

weight               96.86
medical_specialty    49.08
payer_code           39.56
race                  2.23
diag_3                1.40
diag_2                0.35
diag_1                0.02
admission_type_id     0.00
patient_nbr           0.00
encounter_id          0.00
dtype: float64
weight dropped (96.9% missing)


## 5. Removing Patients Who Died

**What I did**
- Read `IDs_mapping.csv` and found the discharge IDs described as "Expired": 11, 19, 20, 21
- Removed those rows (1,652 rows)

**Why**
- A patient who died cannot be readmitted. Keeping these rows would give wrong readmission results.
- I read the IDs from the mapping file instead of typing them, so they come from the official source.
- Only the discharge table has "Expired", so scanning the whole file is safe.

In [49]:
# Find discharge IDs described as "Expired" in the mapping file
expired_ids = []
with open(ID_MAPPING, encoding='utf-8-sig') as file:
    for line in file:
        code, _, description = line.strip().partition(',')
        if code.isdigit() and 'expired' in description.lower():
            expired_ids.append(int(code))
print('Expired IDs:', expired_ids)

# Remove those rows
rows_before = len(df)
df = df[~df['discharge_disposition_id'].isin(expired_ids)].copy()
rows_expired_removed = rows_before - len(df)
print('Rows removed (deceased):', rows_expired_removed)

Expired IDs: [11, 19, 20, 21]
Rows removed (deceased): 1652


## 6. Removing Duplicates

**What I did**
- Counted rows that are exactly the same in every column
- Dropped them (result: 0 duplicates found)

**Why**
- An identical row is a data entry mistake and would count the same visit twice.
- I kept different visits of the same patient, because those are real.

In [50]:
# Count and drop rows that are identical in every column
n_duplicates = int(df.duplicated().sum())
print('Exact duplicate rows:', n_duplicates)

df = df.drop_duplicates()

Exact duplicate rows: 0


## 7. Fixing Data Types

**What I did:** Changed the 5 ID columns from integer to `category`.

**Why:** IDs are labels, not quantities. As categories, nobody can do meaningless maths on them, like the average of `patient_nbr`.

In [51]:
# ID columns are categories, not numbers
ID_COLUMNS = ['encounter_id', 'patient_nbr', 'admission_type_id',
              'discharge_disposition_id', 'admission_source_id']

for col in ID_COLUMNS:
    df[col] = df[col].astype('category')

df[ID_COLUMNS].dtypes

encounter_id                category
patient_nbr                 category
admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
dtype: object

## 8. Saving the Clean Data

**What I did**
- Printed a short summary: 101,766 rows at the start, 1,652 deceased rows removed, 0 duplicates, so 100,114 rows are left (49 columns)
- Saved the data to `01_cleaned.csv`

**Why:** The summary shows what was removed at each step, and the saved file is the input for Phase 2.

In [52]:
# Short summary of the cleaning
print('Raw rows:', rows_raw)
print('Deceased rows removed:', rows_expired_removed)
print('Duplicate rows removed:', n_duplicates)
print('Clean shape:', df.shape)

# Save cleaned data
df.to_csv(OUT_CLEANED, index=False)

Raw rows: 101766
Deceased rows removed: 1652
Duplicate rows removed: 0
Clean shape: (100114, 49)
